# ContosoRetailDW — Data Profile
SQL-based profiling: all stats computed inside SQLite, zero full-table loads into memory.

In [ ]:
import sqlite3
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

DB_PATH = os.path.join(os.path.dirname(os.getcwd()), 'contoso.db') \
          if os.path.basename(os.getcwd()) == 'notebooks' \
          else 'contoso.db'

conn = sqlite3.connect(DB_PATH)
print(f'Connected: {DB_PATH}')

## Schema Summary

In [ ]:
tables = [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"
).fetchall()]

rows = []
for t in tables:
    n_rows = conn.execute(f'SELECT COUNT(*) FROM [{t}];').fetchone()[0]
    cols   = conn.execute(f'PRAGMA table_info([{t}]);').fetchall()
    rows.append({'Table': t, 'Rows': n_rows, 'Columns': len(cols)})

summary = pd.DataFrame(rows)
summary['% of Rows'] = (summary['Rows'] / summary['Rows'].sum() * 100).round(1)
summary = summary.sort_values('Rows', ascending=False).reset_index(drop=True)

summary.style \
    .format({'Rows': '{:,}', '% of Rows': '{:.1f}%'}) \
    .bar(subset=['% of Rows'], color='#4a90d9', vmin=0, vmax=100)

## Row Distribution

In [ ]:
plot_df = summary[summary['Rows'] > 0].copy()

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#4a90d9' if 'Fact' in t else '#a0c4e8' for t in plot_df['Table']]
bars = ax.barh(plot_df['Table'], plot_df['Rows'], color=colors)

ax.set_xlabel('Row Count')
ax.set_title('Row Counts by Table  (blue = Fact, light = Dimension/Other)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.invert_yaxis()
ax.bar_label(bars, labels=[f'{v:,}' for v in plot_df['Rows']], padding=4, fontsize=8)
plt.tight_layout()
plt.show()

## Column Profile
Set `TARGET` to a specific table name, or `None` to profile all tables.

In [ ]:
TARGET = None   # e.g. 'FactSales' or None for all

FREQ_CARD_LIMIT = 50
TOP_N = 5

def is_numeric(t): return any(k in t for k in ('INT','REAL','NUMERIC','FLOAT','DOUBLE'))
def is_date(name, t): return 'date' in name.lower() and 'TEXT' in t

def profile_table(table):
    cols   = conn.execute(f'PRAGMA table_info([{table}]);').fetchall()
    n_rows = conn.execute(f'SELECT COUNT(*) FROM [{table}];').fetchone()[0]
    print(f'\n━━━ {table}  ({n_rows:,} rows × {len(cols)} columns) ━━━')

    if n_rows == 0:
        print('  [empty]'); return

    # ── Column overview ──────────────────────────────────────
    col_rows = []
    num_cols, date_cols, text_cols = [], [], []
    for r in cols:
        col, ctype = r[1], r[2].upper()
        nulls    = conn.execute(f'SELECT COUNT(*) FROM [{table}] WHERE [{col}] IS NULL;').fetchone()[0]
        distinct = conn.execute(f'SELECT COUNT(DISTINCT [{col}]) FROM [{table}];').fetchone()[0]
        col_rows.append({'Column': col, 'Type': ctype,
                         'Nulls': nulls, '% Null': round(nulls/n_rows*100,1),
                         'Distinct': distinct})
        if is_numeric(ctype):           num_cols.append(col)
        elif is_date(col, ctype):       date_cols.append(col)
        elif 1 < distinct <= FREQ_CARD_LIMIT: text_cols.append(col)

    display(pd.DataFrame(col_rows).style.format({'Nulls': '{:,}', '% Null': '{:.1f}', 'Distinct': '{:,}'})
            .bar(subset=['% Null'], color='#f4a261', vmin=0, vmax=100))

    # ── Numeric stats ────────────────────────────────────────
    if num_cols:
        num_rows = []
        for col in num_cols:
            mn, mx, avg = conn.execute(
                f'SELECT MIN([{col}]), MAX([{col}]), AVG([{col}]) FROM [{table}];'
            ).fetchone()
            num_rows.append({'Column': col, 'Min': mn, 'Avg': avg, 'Max': mx})
        print('\n  Numeric stats:')
        display(pd.DataFrame(num_rows).style.format(
            {'Min': '{:,.2f}', 'Avg': '{:,.2f}', 'Max': '{:,.2f}'}, na_rep='NULL'))

    # ── Date ranges ──────────────────────────────────────────
    if date_cols:
        date_rows = []
        for col in date_cols:
            mn, mx = conn.execute(f'SELECT MIN([{col}]), MAX([{col}]) FROM [{table}];').fetchone()
            date_rows.append({'Column': col, 'Min': mn, 'Max': mx})
        print('\n  Date ranges:')
        display(pd.DataFrame(date_rows))

    # ── Top values ───────────────────────────────────────────
    if text_cols:
        print(f'\n  Top {TOP_N} values (low-cardinality columns):')
        for col in text_cols:
            top = conn.execute(f'''
                SELECT [{col}], COUNT(*) AS cnt FROM [{table}]
                WHERE [{col}] IS NOT NULL
                GROUP BY [{col}] ORDER BY cnt DESC LIMIT {TOP_N};
            ''').fetchall()
            if top:
                df = pd.DataFrame(top, columns=[col, 'Count'])
                df['%'] = (df['Count'] / n_rows * 100).round(1)
                display(df.style.format({'Count': '{:,}', '%': '{:.1f}'}).hide(axis='index'))

to_profile = [TARGET] if TARGET else tables
for t in to_profile:
    profile_table(t)